In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, 
    roc_curve, 
    roc_auc_score, 
    accuracy_score,
    precision_recall_curve,
    average_precision_score
)
import warnings
warnings.filterwarnings('ignore')

class DatasetQualityAnalyzer:
    """
    Random Forest classifier for predicting dataset quality.
    """
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.model = None
        self.scaler = StandardScaler()
        self.feature_names = None
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        
    def load_data(self, filepath='quality_matrix.csv'):
        """Load and prepare the quality matrix data."""
        print("=" * 80)
        print("DATASET QUALITY ANALYZER - RANDOM FOREST CLASSIFIER")
        print("=" * 80)
        
        try:
            df = pd.read_csv(filepath)
            print(f"\n✓ Loaded data from '{filepath}'")
            print(f"  Total samples: {len(df)}")
        except FileNotFoundError:
            print(f"\n❌ Error: '{filepath}' not found!")
            return None
        
        # Separate features and target
        X = df.drop(columns=['dataset_name', 'quality_label'])
        y = (df['quality_label'] == 'Bad').astype(int)  # 1 = Bad, 0 = Good
        
        # Store feature names
        self.feature_names = X.columns.tolist()
        
        # Handle missing values
        print(f"\nHandling missing values...")
        missing_before = X.isnull().sum().sum()
        X = X.fillna(X.median())  # Fill with median for numerical stability
        print(f"  Filled {missing_before} missing values with median")
        
        # Check class distribution
        print(f"\nClass distribution:")
        for label, count in zip(['Good', 'Bad'], [(y == 0).sum(), (y == 1).sum()]):
            pct = count / len(y) * 100
            print(f"  {label}: {count} ({pct:.1f}%)")
        
        return X, y
    
    def train_model(self, X, y, test_size=0.2, n_estimators=100, max_depth=10):
        """Train Random Forest model with cross-validation."""
        print("\n" + "=" * 80)
        print("MODEL TRAINING")
        print("=" * 80)
        
        # Split data
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=test_size, random_state=self.random_state, stratify=y
        )
        
        print(f"\nData split:")
        print(f"  Training set: {len(self.X_train)} samples")
        print(f"  Test set: {len(self.X_test)} samples")
        
        # Scale features
        print(f"\nScaling features...")
        self.X_train_scaled = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled = self.scaler.transform(self.X_test)
        print(f"  ✓ Features standardized (mean=0, std=1)")
        
        # Initialize Random Forest with class weights
        print(f"\nTraining Random Forest...")
        print(f"  n_estimators: {n_estimators}")
        print(f"  max_depth: {max_depth}")
        print(f"  class_weight: balanced")
        
        self.model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            class_weight='balanced',
            random_state=self.random_state,
            n_jobs=-1
        )
        
        self.model.fit(self.X_train_scaled, self.y_train)
        print(f"  ✓ Model trained successfully")
        
        # Cross-validation
        print(f"\nPerforming 5-fold cross-validation...")
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=self.random_state)
        cv_scores = cross_val_score(
            self.model, self.X_train_scaled, self.y_train, 
            cv=cv, scoring='accuracy'
        )
        
        print(f"  CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
        
        return self.model
    
    def evaluate_model(self):
        """Evaluate model performance."""
        print("\n" + "=" * 80)
        print("MODEL EVALUATION")
        print("=" * 80)
        
        # Predictions
        y_pred = self.model.predict(self.X_test_scaled)
        y_pred_proba = self.model.predict_proba(self.X_test_scaled)[:, 1]
        
        # Accuracy
        accuracy = accuracy_score(self.y_test, y_pred)
        print(f"\nTest Accuracy: {accuracy:.4f}")
        
        # ROC-AUC
        roc_auc = roc_auc_score(self.y_test, y_pred_proba)
        print(f"ROC-AUC Score: {roc_auc:.4f}")
        
        # Classification Report
        print(f"\nClassification Report:")
        print("-" * 80)
        report = classification_report(
            self.y_test, y_pred, 
            target_names=['Good', 'Bad'],
            digits=4
        )
        print(report)
        
        return {
            'accuracy': accuracy,
            'roc_auc': roc_auc,
            'y_pred': y_pred,
            'y_pred_proba': y_pred_proba
        }
    
    def plot_roc_curve(self, save_path='model_roc_curve.png'):
        """Plot smoothed ROC curve using isotonic regression for interpolation."""
        from sklearn.isotonic import IsotonicRegression
        
        # Get predictions on test set
        y_pred_proba = self.model.predict_proba(self.X_test_scaled)[:, 1]
        
        # Calculate ROC curve
        fpr, tpr, thresholds = roc_curve(self.y_test, y_pred_proba)
        roc_auc = roc_auc_score(self.y_test, y_pred_proba)
        
        # Smooth the ROC curve using isotonic regression
        # Create more points for smoother visualization
        fpr_smooth = np.linspace(0, 1, 300)
        iso_reg = IsotonicRegression(y_min=0, y_max=1, out_of_bounds='clip')
        tpr_smooth = iso_reg.fit_transform(fpr, tpr)
        
        # Interpolate for smooth curve
        tpr_interp = np.interp(fpr_smooth, fpr, tpr_smooth)
        tpr_interp[0] = 0.0
        tpr_interp[-1] = 1.0
        
        # Calculate PR curve
        precision, recall, pr_thresholds = precision_recall_curve(self.y_test, y_pred_proba)
        avg_precision = average_precision_score(self.y_test, y_pred_proba)
        
        # Smooth PR curve
        recall_smooth = np.linspace(0, 1, 300)
        precision_interp = np.interp(recall_smooth, recall[::-1], precision[::-1])
        
        # Create figure
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        # ============ ROC CURVE ============
        # Plot original points
        axes[0].plot(fpr, tpr, 'o', color='lightblue', markersize=4, 
                     alpha=0.5, label='Actual predictions')
        
        # Plot smoothed curve
        axes[0].plot(fpr_smooth, tpr_interp, color='#2E86DE', lw=3, 
                     label=f'ROC Curve (AUC = {roc_auc:.4f})')
        
        axes[0].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', 
                     label='Random Classifier (AUC = 0.5)')
        
        # Find optimal threshold (Youden's J statistic)
        optimal_idx = np.argmax(tpr - fpr)
        optimal_threshold = thresholds[optimal_idx]
        axes[0].scatter(fpr[optimal_idx], tpr[optimal_idx], 
                       s=200, c='red', marker='o', 
                       label=f'Optimal Threshold = {optimal_threshold:.3f}',
                       edgecolors='darkred', linewidths=2, zorder=5)
        
        axes[0].set_xlim([-0.02, 1.02])
        axes[0].set_ylim([-0.02, 1.02])
        axes[0].set_xlabel('False Positive Rate', fontsize=13, fontweight='bold')
        axes[0].set_ylabel('True Positive Rate', fontsize=13, fontweight='bold')
        axes[0].set_title('Receiver Operating Characteristic (ROC) Curve', 
                         fontsize=14, fontweight='bold', pad=15)
        axes[0].legend(loc="lower right", fontsize=9, framealpha=0.95)
        axes[0].grid(True, alpha=0.3, linestyle='--', linewidth=0.7)
        axes[0].set_aspect('equal')
        
        # Add performance warning if AUC is suspiciously high
        if roc_auc > 0.95:
            axes[0].text(0.5, 0.05, '⚠ High AUC - Check for overfitting', 
                        fontsize=9, color='orange', fontweight='bold', 
                        ha='center', bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))
        
        # ============ PRECISION-RECALL CURVE ============
        # Plot original points
        axes[1].plot(recall, precision, 'o', color='lightgreen', markersize=4,
                     alpha=0.5, label='Actual predictions')
        
        # Plot smoothed curve
        axes[1].plot(recall_smooth, precision_interp, color='#10AC84', lw=3,
                     label=f'PR Curve (AP = {avg_precision:.4f})')
        
        # Baseline (proportion of positive class)
        baseline = self.y_test.sum() / len(self.y_test)
        axes[1].plot([0, 1], [baseline, baseline], color='gray', lw=2, 
                    linestyle='--', label=f'Baseline (No Skill = {baseline:.3f})')
        
        axes[1].set_xlim([-0.02, 1.02])
        axes[1].set_ylim([-0.02, 1.02])
        axes[1].set_xlabel('Recall (Sensitivity)', fontsize=13, fontweight='bold')
        axes[1].set_ylabel('Precision', fontsize=13, fontweight='bold')
        axes[1].set_title('Precision-Recall Curve', 
                         fontsize=14, fontweight='bold', pad=15)
        axes[1].legend(loc="lower left", fontsize=9, framealpha=0.95)
        axes[1].grid(True, alpha=0.3, linestyle='--', linewidth=0.7)
        
        # Add sample size warning
        fig.text(0.5, 0.02, f'Note: Test set size = {len(self.y_test)} samples | Consider larger dataset for more reliable curves', 
                ha='center', fontsize=9, style='italic', color='gray')
        
        plt.tight_layout(rect=[0, 0.03, 1, 1])
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"\n✓ ROC curves saved to '{save_path}'")
        print(f"\n⚠ Warning: Small test set ({len(self.y_test)} samples) causes stepped curves")
        print(f"   Recommendation: Collect more data or use cross-validation for smoother curves")
        plt.close()
        
        return roc_auc
    
    def plot_feature_importance(self, top_n=10, save_path='feature_importance.png'):
        """Plot feature importance from Random Forest."""
        importances = self.model.feature_importances_
        indices = np.argsort(importances)[::-1][:top_n]
        
        plt.figure(figsize=(10, 6))
        plt.barh(range(top_n), importances[indices], color='steelblue', alpha=0.8)
        plt.yticks(range(top_n), [self.feature_names[i] for i in indices])
        plt.xlabel('Feature Importance', fontsize=12)
        plt.title(f'Top {top_n} Most Important Features', fontsize=14, fontweight='bold')
        plt.gca().invert_yaxis()
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"✓ Feature importance plot saved to '{save_path}'")
        plt.close()
        
        # Print feature importance
        print(f"\nTop {top_n} Feature Importances:")
        print("-" * 80)
        for i, idx in enumerate(indices, 1):
            print(f"  {i}. {self.feature_names[idx]:<30} {importances[idx]:.4f}")
    
    def analyze_predictions(self):
        """Analyze model predictions in detail."""
        print("\n" + "=" * 80)
        print("PREDICTION ANALYSIS")
        print("=" * 80)
        
        y_pred_proba = self.model.predict_proba(self.X_test_scaled)[:, 1]
        
        # Prediction confidence distribution
        print(f"\nPrediction confidence distribution:")
        print(f"  High confidence (>0.8): {(y_pred_proba > 0.8).sum()} samples")
        print(f"  Medium confidence (0.5-0.8): {((y_pred_proba >= 0.5) & (y_pred_proba <= 0.8)).sum()} samples")
        print(f"  Low confidence (<0.5): {(y_pred_proba < 0.5).sum()} samples")
        
        # Find borderline cases
        borderline_mask = (y_pred_proba > 0.4) & (y_pred_proba < 0.6)
        n_borderline = borderline_mask.sum()
        print(f"\nBorderline cases (0.4 < prob < 0.6): {n_borderline} samples")
        
        if n_borderline > 0:
            print(f"  These may require manual review")


def main():
    """Main execution function."""
    
    # Initialize analyzer
    analyzer = DatasetQualityAnalyzer(random_state=42)
    
    # Load data
    X, y = analyzer.load_data('quality_matrix.csv')
    if X is None:
        return
    
    # Train model
    analyzer.train_model(X, y, test_size=0.2, n_estimators=100, max_depth=10)
    
    # Evaluate model
    results = analyzer.evaluate_model()
    
    # Plot ROC curve (clean, single curve)
    roc_auc = analyzer.plot_roc_curve()
    
    # Plot feature importance
    analyzer.plot_feature_importance(top_n=10)
    
    # Analyze predictions
    analyzer.analyze_predictions()
    
    print("\n" + "=" * 80)
    print("✓ MODEL TRAINING AND EVALUATION COMPLETE")
    print("=" * 80)
    print(f"\nModel Summary:")
    print(f"  • Test Accuracy: {results['accuracy']:.4f}")
    print(f"  • ROC-AUC Score: {results['roc_auc']:.4f}")
    print(f"  • Model saved in memory (ready for predictions)")
    print(f"\nGenerated files:")
    print(f"  • model_roc_curve.png - Clean ROC and Precision-Recall curves")
    print(f"  • feature_importance.png - Top feature importances")
    
    return analyzer


if __name__ == '__main__':
    analyzer = main()

DATASET QUALITY ANALYZER - RANDOM FOREST CLASSIFIER

✓ Loaded data from 'quality_matrix.csv'
  Total samples: 200

Handling missing values...
  Filled 184 missing values with median

Class distribution:
  Good: 118 (59.0%)
  Bad: 82 (41.0%)

MODEL TRAINING

Data split:
  Training set: 160 samples
  Test set: 40 samples

Scaling features...
  ✓ Features standardized (mean=0, std=1)

Training Random Forest...
  n_estimators: 100
  max_depth: 10
  class_weight: balanced
  ✓ Model trained successfully

Performing 5-fold cross-validation...
  CV Accuracy: 0.9625 (+/- 0.0612)

MODEL EVALUATION

Test Accuracy: 0.9750
ROC-AUC Score: 1.0000

Classification Report:
--------------------------------------------------------------------------------
              precision    recall  f1-score   support

        Good     1.0000    0.9583    0.9787        24
         Bad     0.9412    1.0000    0.9697        16

    accuracy                         0.9750        40
   macro avg     0.9706    0.9792    